## Embdeddings and Vectorstores 

In [1]:
import os
import openai
import sys
sys.path.append('../..')

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.environ['OPENAI_API_KEY']

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# Load PDF
loaders = [
    # Duplicate documents on purpose - messy data
    PyPDFLoader("docs/MachineLearning-Lecture01.pdf"),
    PyPDFLoader("docs/MachineLearning-Lecture01.pdf"),
    PyPDFLoader("docs/MachineLearning-Lecture02.pdf"),
    PyPDFLoader("docs/MachineLearning-Lecture03.pdf")
]
docs = []
for loader in loaders:
    docs.extend(loader.load())

In [3]:
# Split
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 150
)

In [4]:
splits = text_splitter.split_documents(docs)

In [5]:
len(splits)

208

## Embeddings

In [6]:
# !pip install langchain_openai

In [7]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings()

In [8]:
sentence1 = "i like dogs"
sentence2 = "i like canines"
sentence3 = "the weather is ugly outside"

In [9]:
embedding1 = embedding.embed_query(sentence1)
embedding2 = embedding.embed_query(sentence2)
embedding3 = embedding.embed_query(sentence3)

In [10]:
len(embedding1)

1536

In [11]:
embedding1[:240]

[-0.02755114622414112,
 -0.005382069852203131,
 -0.02582130767405033,
 -0.03305632621049881,
 -0.027349121868610382,
 0.02263941615819931,
 -0.01043584942817688,
 -0.008125188760459423,
 0.0025079497136175632,
 -0.01964692212641239,
 0.0006735478527843952,
 0.029243104159832,
 -0.005397852975875139,
 0.0007114275358617306,
 0.00021386229491326958,
 0.014078610576689243,
 0.02987443283200264,
 -0.001194787910208106,
 0.004210956394672394,
 -0.0038700394798070192,
 -0.01169219147413969,
 0.006875160150229931,
 0.012992726638913155,
 -0.04724857583642006,
 -0.002298033330589533,
 0.004706548992544413,
 0.016932211816310883,
 -0.0001317896822001785,
 -0.025884440168738365,
 -0.01635139063000679,
 0.02669254131615162,
 0.0029972288757562637,
 -0.015518037602305412,
 -0.02476067654788494,
 0.006035494152456522,
 -0.014949843287467957,
 0.00978558138012886,
 -0.01151541993021965,
 -0.004349848721176386,
 -0.010877778753638268,
 -0.0170584786683321,
 0.01118081621825695,
 0.006938292644917965,

In [12]:
import numpy as np

np.dot(embedding1, embedding2)

0.9631511809630344

In [13]:
np.dot(embedding1, embedding3)

0.770203137103822

In [14]:
np.dot(embedding2, embedding3)

0.759054062979165

In [15]:
# ! pip install chromadb

In [16]:
from langchain_community.vectorstores import Chroma

In [17]:
rm -rf docs/chroma/

In [18]:
persist_directory = 'docs/chroma/'

In [19]:
import os
os.makedirs(persist_directory, exist_ok=True)

In [20]:
# !pip uninstall chromadb -y
# !pip install chromadb==0.4.24

In [21]:
import sys
print(sys.executable)

/Users/koushalsmodi/Desktop/MachineLearning/MachineLearningProjects/NLP/.venv/bin/python


In [22]:
vectordb = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    persist_directory=persist_directory
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
print(vectordb._collection.count())
# number of vectors stored in your vector database

208


## Similarity Search

In [24]:
question = "is there an email i can ask for help"

In [25]:
docs = vectordb.similarity_search(question,k=3)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [26]:
len(docs)

3

In [27]:
docs[0].page_content

"cs229-qa@cs.stanford.edu. This goes to an account that's read by all the TAs and me. So \nrather than sending us email individually, if you send email to this account, it will \nactually let us get back to you maximally quickly with answers to your questions.  \nIf you're asking questions about homework problems, please say in the subject line which \nassignment and which question the email refers to, since that will also help us to route \nyour question to the appropriate TA or to me appropriately and get the response back to \nyou quickly.  \nLet's see. Skipping ahead — let's see — for homework, one midterm, one open and term \nproject. Notice on the honor code. So one thing that I think will help you to succeed and \ndo well in this class and even help you to enjoy this class more is if you form a study \ngroup.  \nSo start looking around where you're sitting now or at the end of class today, mingle a \nlittle bit and get to know your classmates. I strongly encourage you to form st

In [28]:
vectordb.persist()

/var/folders/1_/bwh3j9hx0fs5c8dhd9d9l_zc0000gn/T/ipykernel_65418/3711397106.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


## Failure modes

In [29]:
question = "what did they say about matlab?"

In [30]:
docs = vectordb.similarity_search(question,k=5)

In [31]:
# Notice that we're getting duplicate chunks (because of the duplicate MachineLearning-Lecture01.pdf in the index).
# Semantic search fetches all similar documents, but does not enforce diversity docs[0] and docs[1] are indentical.

In [32]:
docs[0].page_content

'those homeworks will be done in either MATLAB or in Octave, which is sort of — I \nknow some people call it a free version of MATLAB, which it sort of is, sort of isn\'t.  \nSo I guess for those of you that haven\'t seen MATLAB before, and I know most of you \nhave, MATLAB is I guess part of the programming language that makes it very easy to \nwrite codes using matrices, to write code for numerical routines, to move data around, to \nplot data. And it\'s sort of an extremely easy to learn tool to use for implementing a lot of \nlearning algorithms.  \nAnd in case some of you want to work on your own home computer or something if you \ndon\'t have a MATLAB license, for the purposes of this class, there\'s also — [inaudible] \nwrite that down [inaudible] MATLAB — there\'s also a software package called Octave \nthat you can download for free off the Internet. And it has somewhat fewer features than \nMATLAB, but it\'s free, and for the purposes of this class, it will work for just abou

In [33]:
docs[1].page_content

'those homeworks will be done in either MATLAB or in Octave, which is sort of — I \nknow some people call it a free version of MATLAB, which it sort of is, sort of isn\'t.  \nSo I guess for those of you that haven\'t seen MATLAB before, and I know most of you \nhave, MATLAB is I guess part of the programming language that makes it very easy to \nwrite codes using matrices, to write code for numerical routines, to move data around, to \nplot data. And it\'s sort of an extremely easy to learn tool to use for implementing a lot of \nlearning algorithms.  \nAnd in case some of you want to work on your own home computer or something if you \ndon\'t have a MATLAB license, for the purposes of this class, there\'s also — [inaudible] \nwrite that down [inaudible] MATLAB — there\'s also a software package called Octave \nthat you can download for free off the Internet. And it has somewhat fewer features than \nMATLAB, but it\'s free, and for the purposes of this class, it will work for just abou

In [34]:
# We can see a new failure mode.
# The question below asks a question about the third lecture, but includes results from other lectures as well.
question = "what did they say about regression in the third lecture?"

In [35]:
docs = vectordb.similarity_search(question,k=5)

In [36]:
for doc in docs:
    print(doc.metadata)

{'author': '', 'creationdate': '2008-07-11T11:25:03-07:00', 'creator': 'PScript5.dll Version 5.2.2', 'moddate': '2008-07-11T11:25:03-07:00', 'page': 0, 'page_label': '1', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'source': 'docs/MachineLearning-Lecture03.pdf', 'title': '', 'total_pages': 16}
{'author': '', 'creationdate': '2008-07-11T11:25:05-07:00', 'creator': 'PScript5.dll Version 5.2.2', 'moddate': '2008-07-11T11:25:05-07:00', 'page': 2, 'page_label': '3', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'source': 'docs/MachineLearning-Lecture02.pdf', 'title': '', 'total_pages': 18}
{'author': '', 'creationdate': '2008-07-11T11:25:05-07:00', 'creator': 'PScript5.dll Version 5.2.2', 'moddate': '2008-07-11T11:25:05-07:00', 'page': 17, 'page_label': '18', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'source': 'docs/MachineLearning-Lecture02.pdf', 'title': '', 'total_pages': 18}
{'author': '', 'creationdate': '2008-07-11T11:25:23-07:00', 'creator': 'PScript5.dll Version 5.2.2

In [37]:
print(docs[4].page_content)

into his office and he said, "Oh, professor, professor, thank you so much for your 
machine learning class. I learned so much from it. There's this stuff that I learned in your 
class, and I now use every day. And it's helped me make lots of money, and here's a 
picture of my big house."  
So my friend was very excited. He said, "Wow. That's great. I'm glad to hear this 
machine learning stuff was actually useful. So what was it that you learned? Was it 
logistic regression? Was it the PCA? Was it the data networks? What was it that you 
learned that was so helpful?" And the student said, "Oh, it was the MATLAB."  
So for those of you that don't know MATLAB yet, I hope you do learn it. It's not hard, 
and we'll actually have a short MATLAB tutorial in one of the discussion sections for 
those of you that don't know it.  
Okay. The very last piece of logistical thing is the discussion sections. So discussion 
sections will be taught by the TAs, and attendance at discussion sections is o